# TP Visión por Computadora II — CEIA FIUBA

## Setup, EDA y Entrenamiento de modelos YOLO para detección de EPP

Esta notebook unifica el setup del dataset, el análisis exploratorio y el entrenamiento
de los tres modelos YOLO. Está pensada para ejecutarse de corrido en Google Colab con GPU.

**Antes de correr:** ir a Runtime → Change runtime type → T4 GPU

In [ ]:
# Instalar dependencias
!pip install -q ultralytics roboflow

In [ ]:
import sys
import platform
import torch
import ultralytics

print(f"Python:      {sys.version}")
print(f"Sistema:     {platform.system()} {platform.machine()}")
print(f"PyTorch:     {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

tiene_gpu = torch.cuda.is_available()
print(f"\nDispositivo: {'CUDA (' + torch.cuda.get_device_name(0) + ')' if tiene_gpu else 'CPU'}")

if not tiene_gpu:
    print("No hay GPU — el entrenamiento va a ser mas lento.")

---
## Parte 1 — Setup y exploración del dataset

### Descarga del dataset

Se utiliza el dataset **Construction Site Safety** de Roboflow Universe (10 clases de EPP).

La API key se configura en el panel de **Secrets** de Colab (icono de llave en el panel izquierdo)
con el nombre `ROBOFLOW_API_KEY`.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

api_key = userdata.get("ROBOFLOW_API_KEY")
if not api_key:
    raise ValueError("Falta la API key de Roboflow. Agregarla en Secrets con el nombre ROBOFLOW_API_KEY.")

carpeta_datos = Path("data")
carpeta_datos.mkdir(exist_ok=True)
print(f"Directorio de datos: {carpeta_datos.resolve()}")

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=api_key)
proyecto = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = proyecto.version(28)

ruta_descarga = str(carpeta_datos / "construction-ppe")
dataset_descargado = version.download("yolov8", location=ruta_descarga)
print(f"\nDescargado en: {dataset_descargado.location}")

In [ ]:
import yaml

ruta_dataset = Path(dataset_descargado.location)

for nombre_split in ["train", "valid", "test"]:
    carpeta_imagenes = ruta_dataset / nombre_split / "images"
    if carpeta_imagenes.exists():
        cantidad = len(list(carpeta_imagenes.glob("*.jpg"))) + len(list(carpeta_imagenes.glob("*.png")))
        print(f"  {nombre_split:6s}: {cantidad:5d} imagenes")

with open(ruta_dataset / "data.yaml") as f:
    config_yaml = yaml.safe_load(f)

nombres_clases = config_yaml["names"]
cantidad_clases = config_yaml["nc"]
print(f"\n{cantidad_clases} clases: {nombres_clases}")

### Análisis exploratorio (EDA)

#### Distribución de clases

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="husl")


def contar_instancias_por_clase(carpeta_labels: Path, nombres_clases: list) -> Counter:
    conteo = Counter()
    for archivo_label in carpeta_labels.glob("*.txt"):
        with open(archivo_label) as f:
            for linea in f:
                id_clase = int(linea.strip().split()[0])
                conteo[nombres_clases[id_clase]] += 1
    return conteo


conteo_por_split = {}
for nombre_split in ["train", "valid", "test"]:
    carpeta_labels = ruta_dataset / nombre_split / "labels"
    if carpeta_labels.exists():
        conteo_por_split[nombre_split] = contar_instancias_por_clase(carpeta_labels, nombres_clases)

paleta_colores = sns.color_palette("husl", len(nombres_clases))
fig, ejes = plt.subplots(1, len(conteo_por_split), figsize=(18, 5))
if len(conteo_por_split) == 1:
    ejes = [ejes]

for eje, (nombre_split, conteo) in zip(ejes, conteo_por_split.items()):
    conteo_ordenado = dict(sorted(conteo.items(), key=lambda x: x[1], reverse=True))
    barras = eje.barh(list(conteo_ordenado.keys()), list(conteo_ordenado.values()), color=paleta_colores)
    eje.set_title(f"{nombre_split} ({sum(conteo.values())} instancias)", fontsize=13, fontweight="bold")
    eje.set_xlabel("Cantidad de instancias")
    for barra, valor in zip(barras, conteo_ordenado.values()):
        eje.text(valor + 20, barra.get_y() + barra.get_height() / 2, str(valor), va="center", fontsize=9)

plt.suptitle("Distribucion de clases por split", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(carpeta_datos / "class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()

**Observaciones:**

- La clase `Person` es la más frecuente en todos los splits (~9600 en train), lo cual es esperable ya que cada trabajador en la imagen genera una anotación de persona además de las anotaciones de EPP.
- Existe un desbalance entre las clases de EPP: por ejemplo, `Mask` tiene aproximadamente la mitad de instancias que `Safety Vest` o `NO-Safety Vest`. Esto podría afectar la capacidad del modelo para detectar mascarillas con la misma precisión que los otros elementos.
- Las clases `vehicle` y `Mask` son las menos representadas en el set de entrenamiento (~1500 instancias cada una).
- La proporción relativa entre clases se mantiene razonablemente consistente entre los tres splits, lo cual indica que la partición fue estratificada.

#### Tamaños de bounding boxes

In [ ]:
def extraer_tamanios_bboxes(carpeta_labels: Path):
    anchos, altos, areas = [], [], []
    for archivo_label in carpeta_labels.glob("*.txt"):
        with open(archivo_label) as f:
            for linea in f:
                partes = linea.strip().split()
                if len(partes) >= 5:
                    ancho_bbox = float(partes[3])
                    alto_bbox = float(partes[4])
                    anchos.append(ancho_bbox)
                    altos.append(alto_bbox)
                    areas.append(ancho_bbox * alto_bbox)
    return anchos, altos, areas


carpeta_labels_train = ruta_dataset / "train" / "labels"
anchos, altos, areas = extraer_tamanios_bboxes(carpeta_labels_train)

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

ejes[0].hist(anchos, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ejes[0].set_title("Ancho (normalizado)")
ejes[0].set_xlabel("Ancho relativo")
ejes[0].set_ylabel("Frecuencia")

ejes[1].hist(altos, bins=50, color="coral", edgecolor="white", alpha=0.8)
ejes[1].set_title("Alto (normalizado)")
ejes[1].set_xlabel("Alto relativo")

ejes[2].hist(areas, bins=50, color="mediumseagreen", edgecolor="white", alpha=0.8)
ejes[2].set_title("Area (ancho x alto)")
ejes[2].set_xlabel("Area relativa")

plt.suptitle(f"Tamanios de bounding boxes — train ({len(anchos):,} instancias)", fontsize=13)
plt.tight_layout()
plt.savefig(carpeta_datos / "bbox_stats.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"\nEstadisticas del area:")
print(f"  Mediana: {np.median(areas):.4f}")
print(f"  Media:   {np.mean(areas):.4f}")
print(f"  Min:     {np.min(areas):.4f}")
print(f"  Max:     {np.max(areas):.4f}")

**Observaciones:**

- La distribución de tamaños está fuertemente sesgada hacia valores pequeños: la mediana del área es ~0.018, mientras que la media es ~0.060. Esto indica que la mayoría de los objetos anotados ocupan menos del 2% de la superficie total de la imagen.
- El sesgo es esperable en imágenes de obras de construcción, donde la cámara suele estar a distancia considerable de los trabajadores y sus elementos de protección.
- Los objetos muy pequeños (área < 0.01) representan una proporción significativa del dataset. Esto tiene implicancias directas en la configuración del entrenamiento: resoluciones de entrada mayores (e.g., 640×640 en lugar de 320×320) permiten al modelo capturar mejor los detalles de estos objetos.
- El rango de áreas es amplio (de ~0.0001 a ~0.53), lo que significa que el modelo deberá manejar simultáneamente objetos muy pequeños (EPP a distancia) y objetos grandes (vehículos o maquinaria en primer plano). Las arquitecturas YOLO abordan esto mediante detección multi-escala con Feature Pyramid Networks (FPN).

#### Visualización de imágenes con anotaciones

In [ ]:
import cv2
import random

COLOR_POR_CLASE = {
    "Hardhat":        (0, 200, 0),
    "Mask":           (0, 180, 0),
    "Safety Vest":    (0, 160, 0),
    "NO-Hardhat":     (220, 0, 0),
    "NO-Mask":        (200, 0, 0),
    "NO-Safety Vest": (180, 0, 0),
    "Person":         (100, 100, 255),
    "Safety Cone":    (255, 165, 0),
    "machinery":      (128, 0, 128),
    "vehicle":        (64, 64, 64),
}


def dibujar_anotaciones_yolo(ruta_imagen: Path, ruta_label: Path, nombres_clases: list) -> np.ndarray:
    imagen = cv2.imread(str(ruta_imagen))
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    alto_img, ancho_img = imagen.shape[:2]

    if ruta_label.exists():
        with open(ruta_label) as f:
            for linea in f:
                partes = linea.strip().split()
                id_clase = int(partes[0])
                centro_x = float(partes[1])
                centro_y = float(partes[2])
                ancho_bbox = float(partes[3])
                alto_bbox = float(partes[4])

                x1 = int((centro_x - ancho_bbox / 2) * ancho_img)
                y1 = int((centro_y - alto_bbox / 2) * alto_img)
                x2 = int((centro_x + ancho_bbox / 2) * ancho_img)
                y2 = int((centro_y + alto_bbox / 2) * alto_img)

                nombre_clase = nombres_clases[id_clase]
                color = COLOR_POR_CLASE.get(nombre_clase, (200, 200, 200))
                cv2.rectangle(imagen, (x1, y1), (x2, y2), color, 2)
                cv2.putText(imagen, nombre_clase, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return imagen


todas_las_imagenes_train = list((ruta_dataset / "train" / "images").glob("*.jpg"))
random.seed(42)
imagenes_muestra = random.sample(todas_las_imagenes_train, min(8, len(todas_las_imagenes_train)))

fig, ejes = plt.subplots(2, 4, figsize=(20, 10))
ejes = ejes.flatten()

for i, ruta_imagen in enumerate(imagenes_muestra):
    ruta_label = ruta_dataset / "train" / "labels" / (ruta_imagen.stem + ".txt")
    imagen_anotada = dibujar_anotaciones_yolo(ruta_imagen, ruta_label, nombres_clases)
    ejes[i].imshow(imagen_anotada)
    ejes[i].axis("off")
    ejes[i].set_title(ruta_imagen.name[:30], fontsize=8)

plt.suptitle("Muestras del dataset (verde = tiene EPP, rojo = le falta EPP)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(carpeta_datos / "sample_annotations.png", bbox_inches="tight", dpi=150)
plt.show()

**Observaciones:**

- Los bounding boxes coinciden correctamente con los objetos en las imágenes, lo cual confirma la calidad de las anotaciones del dataset.
- Se observa que algunas imágenes son mosaicos (composiciones de 4 imágenes en una). Esto es producto de *mosaic augmentation*, una técnica de data augmentation que ya viene aplicada en el dataset. Esta técnica obliga al modelo a detectar objetos en distintas escalas, posiciones y contextos simultáneamente.
- Los objetos de EPP (cascos, chalecos, mascarillas) tienden a ser pequeños en relación al tamaño total de la imagen, consistente con lo observado en el análisis de bounding boxes de la sección anterior.
- Se verifica visualmente la convención de colores: verde para EPP presente (`Hardhat`, `Mask`, `Safety Vest`) y rojo para EPP ausente (`NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`), lo cual facilita la interpretación de las detecciones.

#### Resumen del dataset

In [ ]:
import pandas as pd

filas_resumen = []
for nombre_split in ["train", "valid", "test"]:
    carpeta_imagenes = ruta_dataset / nombre_split / "images"
    carpeta_labels = ruta_dataset / nombre_split / "labels"
    if carpeta_imagenes.exists():
        num_imagenes = len(list(carpeta_imagenes.glob("*.jpg"))) + len(list(carpeta_imagenes.glob("*.png")))
        conteo = contar_instancias_por_clase(carpeta_labels, nombres_clases)
        total_anotaciones = sum(conteo.values())
        filas_resumen.append({
            "Split": nombre_split,
            "Imagenes": num_imagenes,
            "Anotaciones": total_anotaciones,
            "Anotaciones/imagen": round(total_anotaciones / num_imagenes, 1),
        })

tabla_resumen = pd.DataFrame(filas_resumen)
print("=== Resumen del dataset ===")
print(tabla_resumen.to_string(index=False))

In [ ]:
import json

ruta_yaml_dataset = str(ruta_dataset / "data.yaml")

configuracion_dataset = {
    "dataset_yaml": ruta_yaml_dataset,
    "num_classes": cantidad_clases,
    "class_names": nombres_clases,
}

with open(carpeta_datos / "dataset_config.json", "w") as f:
    json.dump(configuracion_dataset, f, indent=2)

print("Configuracion guardada en dataset_config.json")

---
## Parte 2 — Fine-tuning de modelos YOLO

Se entrenan tres variantes de YOLO mediante fine-tuning sobre el dataset Construction-PPE.
Los tres parten de pesos preentrenados en COCO (transfer learning).

### Configuración del entrenamiento

In [ ]:
import shutil
import time
from ultralytics import YOLO

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tamanio_imagen = 640 if dispositivo == "cuda" else 416
tamanio_batch = 16 if dispositivo == "cuda" else 4
cantidad_epocas = 50 if dispositivo == "cuda" else 15
num_workers = 4 if dispositivo == "cuda" else 0

carpeta_runs = Path("runs")
carpeta_modelos = Path("models")
carpeta_modelos.mkdir(exist_ok=True)

print(f"Dispositivo:        {dispositivo}")
print(f"Tamaño de imagen:   {tamanio_imagen}x{tamanio_imagen}")
print(f"Tamaño de batch:    {tamanio_batch}")
print(f"Épocas:             {cantidad_epocas}")

### Entrenamiento de YOLOv8n (Nano)

YOLOv8n es la variante más liviana (~3M parámetros). Se usa `patience=10` para early stopping.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8n (nano)")
print("=" * 60)

modelo_v8n = YOLO("yolov8n.pt")

resultados_v8n = modelo_v8n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

ruta_mejor_v8n = carpeta_runs / "yolov8n_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8n, carpeta_modelos / "yolov8n_best.pt")
print(f"\nYOLOv8n guardado en: {carpeta_modelos / 'yolov8n_best.pt'}")

### Entrenamiento de YOLOv8m (Medium)

YOLOv8m tiene mayor capacidad (~25M parámetros). Batch reducido a la mitad por el mayor tamaño.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8m (medium)")
print("=" * 60)

modelo_v8m = YOLO("yolov8m.pt")

resultados_v8m = modelo_v8m.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=max(2, tamanio_batch // 2),
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8m_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

ruta_mejor_v8m = carpeta_runs / "yolov8m_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8m, carpeta_modelos / "yolov8m_best.pt")
print(f"\nYOLOv8m guardado en: {carpeta_modelos / 'yolov8m_best.pt'}")

### Entrenamiento de YOLOv11n (Nano, arquitectura v11)

Nueva arquitectura con mejoras en el cuello de botella y cabezal de detección.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv11n (YOLO11 nano)")
print("=" * 60)

modelo_v11n = YOLO("yolo11n.pt")

resultados_v11n = modelo_v11n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov11n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

ruta_mejor_v11n = carpeta_runs / "yolov11n_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v11n, carpeta_modelos / "yolov11n_best.pt")
print(f"\nYOLOv11n guardado en: {carpeta_modelos / 'yolov11n_best.pt'}")

### Evaluación en el set de validación

In [ ]:
rutas_modelos = {
    "YOLOv8n": carpeta_modelos / "yolov8n_best.pt",
    "YOLOv8m": carpeta_modelos / "yolov8m_best.pt",
    "YOLOv11n": carpeta_modelos / "yolov11n_best.pt",
}

resultados_evaluacion = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        print(f"  {nombre_modelo}: no encontrado, se omite.")
        continue

    print(f"\nEvaluando {nombre_modelo}...")
    modelo = YOLO(str(ruta_modelo))

    inicio = time.time()
    metricas = modelo.val(
        data=ruta_yaml_dataset,
        imgsz=tamanio_imagen,
        device=dispositivo,
        workers=num_workers,
        verbose=False,
    )
    tiempo_evaluacion = time.time() - inicio

    resultados_evaluacion[nombre_modelo] = {
        "map50": round(float(metricas.box.map50), 4),
        "map50_95": round(float(metricas.box.map), 4),
        "precision": round(float(metricas.box.mp), 4),
        "recall": round(float(metricas.box.mr), 4),
        "val_time_s": round(tiempo_evaluacion, 1),
    }

    r = resultados_evaluacion[nombre_modelo]
    print(f"  mAP50:      {r['map50']:.4f}")
    print(f"  mAP50-95:   {r['map50_95']:.4f}")
    print(f"  Precision:  {r['precision']:.4f}")
    print(f"  Recall:     {r['recall']:.4f}")

with open(carpeta_datos / "eval_results.json", "w") as f:
    json.dump(resultados_evaluacion, f, indent=2)

print("\nResultados guardados en data/eval_results.json")

### Benchmark de latencia de inferencia

In [ ]:
imagen_sintetica = np.random.randint(0, 255, (tamanio_imagen, tamanio_imagen, 3), dtype=np.uint8)
cantidad_mediciones = 20

resultados_latencia = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        continue

    modelo = YOLO(str(ruta_modelo))

    for _ in range(3):
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)

    tiempos_ms = []
    for _ in range(cantidad_mediciones):
        t0 = time.perf_counter()
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)
        tiempos_ms.append((time.perf_counter() - t0) * 1000)

    resultados_latencia[nombre_modelo] = {
        "mean_ms": round(np.mean(tiempos_ms), 1),
        "std_ms": round(np.std(tiempos_ms), 1),
        "p50_ms": round(np.percentile(tiempos_ms, 50), 1),
        "p95_ms": round(np.percentile(tiempos_ms, 95), 1),
        "fps": round(1000 / np.mean(tiempos_ms), 1),
    }

    r = resultados_latencia[nombre_modelo]
    print(f"{nombre_modelo}: {r['mean_ms']:.1f} ms +/- {r['std_ms']:.1f} | {r['fps']:.1f} FPS")

with open(carpeta_datos / "latency_results.json", "w") as f:
    json.dump(resultados_latencia, f, indent=2)

print("\nLatencias guardadas en data/latency_results.json")

In [ ]:
from google.colab import files

archivos_a_descargar = [
    carpeta_modelos / "yolov8n_best.pt",
    carpeta_modelos / "yolov8m_best.pt",
    carpeta_modelos / "yolov11n_best.pt",
    carpeta_datos / "eval_results.json",
    carpeta_datos / "latency_results.json",
    carpeta_datos / "dataset_config.json",
]

for archivo in archivos_a_descargar:
    if archivo.exists():
        print(f"Descargando {archivo.name}...")
        files.download(str(archivo))
    else:
        print(f"  {archivo.name} no encontrado, se omite.")

In [ ]:
import matplotlib.patches as mpatches

# Cargar resultados generados en la Parte 2
with open(carpeta_datos / "eval_results.json") as f:
    resultados_evaluacion = json.load(f)

with open(carpeta_datos / "latency_results.json") as f:
    resultados_latencia = json.load(f)

# Información de complejidad por modelo
info_modelos = {
    "YOLOv8n":  {"params_M": 3.2,  "flops_G": 8.7},
    "YOLOv8m":  {"params_M": 25.9, "flops_G": 78.9},
    "YOLOv11n": {"params_M": 2.6,  "flops_G": 6.5},
}

print("Resultados cargados:")
print(json.dumps(resultados_evaluacion, indent=2))

### Tabla comparativa general

In [ ]:
filas_comparacion = []
for nombre_modelo in resultados_evaluacion:
    fila = {"Modelo": nombre_modelo}
    fila.update(resultados_evaluacion[nombre_modelo])
    if nombre_modelo in resultados_latencia:
        fila.update(resultados_latencia[nombre_modelo])
    if nombre_modelo in info_modelos:
        fila.update(info_modelos[nombre_modelo])
    filas_comparacion.append(fila)

tabla_comparacion = pd.DataFrame(filas_comparacion).set_index("Modelo")

columnas_display = ["map50", "map50_95", "precision", "recall", "mean_ms", "fps", "params_M", "flops_G"]
nombres_columnas = ["mAP50", "mAP50-95", "Precision", "Recall", "Latencia (ms)", "FPS", "Params (M)", "FLOPs (G)"]
tabla_display = tabla_comparacion[columnas_display].copy()
tabla_display.columns = nombres_columnas

print("=" * 70)
print("           COMPARACION DE MODELOS - Construction PPE Detection")
print("=" * 70)
print(tabla_display.to_string())

### Visualización de métricas de detección

In [ ]:
nombres_modelos = list(resultados_evaluacion.keys())
posiciones_x = np.arange(len(nombres_modelos))
colores_modelos = ["#2196F3", "#FF5722", "#4CAF50"]

fig, ejes = plt.subplots(2, 2, figsize=(14, 10))

metricas_a_graficar = [
    ("map50",    "mAP@50",    ejes[0, 0]),
    ("map50_95", "mAP@50-95", ejes[0, 1]),
    ("precision", "Precision", ejes[1, 0]),
    ("recall",   "Recall",     ejes[1, 1]),
]

for clave_metrica, etiqueta_metrica, eje in metricas_a_graficar:
    valores = [resultados_evaluacion[m].get(clave_metrica, 0) for m in nombres_modelos]
    barras = eje.bar(nombres_modelos, valores, color=colores_modelos[:len(nombres_modelos)],
                     edgecolor="white", linewidth=1.5)
    eje.set_title(etiqueta_metrica, fontsize=13, fontweight="bold")
    eje.set_ylim(0, 1.0)
    eje.set_ylabel("Score")
    eje.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)
    for barra, val in zip(barras, valores):
        eje.text(barra.get_x() + barra.get_width() / 2, val + 0.01, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.suptitle("Comparacion de metricas de deteccion - Construction-PPE Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(carpeta_datos / "comparison_metrics.png", bbox_inches="tight", dpi=150)
plt.show()

### Trade-off velocidad vs. precisión

El tamaño del punto representa la cantidad de parámetros del modelo. Las líneas punteadas
delimitan la zona de despliegue práctico (FPS > 15 y mAP50 > 0.6).

In [ ]:
fig, eje = plt.subplots(figsize=(9, 6))

for i, nombre_modelo in enumerate(nombres_modelos):
    if nombre_modelo not in resultados_latencia:
        continue
    fps_modelo = resultados_latencia[nombre_modelo]["fps"]
    map50_modelo = resultados_evaluacion[nombre_modelo]["map50"]
    params_modelo = info_modelos.get(nombre_modelo, {}).get("params_M", 5)

    eje.scatter(fps_modelo, map50_modelo, s=params_modelo * 20, color=colores_modelos[i],
                alpha=0.85, edgecolors="black", linewidth=1.5, zorder=5)
    eje.annotate(f"{nombre_modelo}\n({params_modelo}M params)",
                 xy=(fps_modelo, map50_modelo), xytext=(10, 5), textcoords="offset points",
                 fontsize=10, fontweight="bold", color=colores_modelos[i])

eje.set_xlabel("FPS (frames por segundo) - mayor es mejor", fontsize=12)
eje.set_ylabel("mAP@50 - mayor es mejor", fontsize=12)
eje.set_title("Trade-off: Velocidad vs. Precision\n(tamanio del punto = cantidad de parametros)", fontsize=13)
eje.grid(True, alpha=0.3)

eje.axvline(x=15, color="orange", linestyle="--", alpha=0.5, label="15 FPS minimo")
eje.axhline(y=0.6, color="green", linestyle="--", alpha=0.5, label="0.6 mAP50 objetivo")
eje.legend(fontsize=10)

plt.tight_layout()
plt.savefig(carpeta_datos / "speed_accuracy_tradeoff.png", bbox_inches="tight", dpi=150)
plt.show()

### Curvas de entrenamiento

Se comparan las curvas de mAP@50 y box loss (train y validación) para cada modelo.

In [ ]:
directorios_runs = {
    "YOLOv8n":  carpeta_runs / "yolov8n_ppe",
    "YOLOv8m":  carpeta_runs / "yolov8m_ppe",
    "YOLOv11n": carpeta_runs / "yolov11n_ppe",
}

colores_curvas = {"YOLOv8n": "#2196F3", "YOLOv8m": "#FF5722", "YOLOv11n": "#4CAF50"}

fig, ejes = plt.subplots(1, 3, figsize=(18, 5))
metricas_curvas = ["metrics/mAP50(B)", "train/box_loss", "val/box_loss"]
etiquetas_curvas = ["mAP@50 (val)", "Box Loss (train)", "Box Loss (val)"]

for eje, metrica, etiqueta in zip(ejes, metricas_curvas, etiquetas_curvas):
    for nombre_modelo, dir_run in directorios_runs.items():
        ruta_csv = dir_run / "results.csv"
        if not ruta_csv.exists():
            print(f"No encontrado: {ruta_csv}")
            continue
        df_run = pd.read_csv(ruta_csv)
        df_run.columns = [c.strip() for c in df_run.columns]
        if metrica in df_run.columns:
            eje.plot(df_run["epoch"], df_run[metrica],
                     label=nombre_modelo, color=colores_curvas[nombre_modelo], linewidth=2)
    eje.set_xlabel("Epoca")
    eje.set_title(etiqueta, fontsize=12, fontweight="bold")
    eje.legend()
    eje.grid(alpha=0.3)

plt.suptitle("Curvas de entrenamiento por modelo", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(carpeta_datos / "training_curves.png", bbox_inches="tight", dpi=150)
plt.show()

### Métricas por clase del mejor modelo

Se re-evalúa el modelo con mayor mAP50 para obtener el AP@50 desglosado por clase.

In [ ]:
# Determinar el mejor modelo segun mAP50
nombre_mejor_modelo = max(resultados_evaluacion, key=lambda m: resultados_evaluacion[m].get("map50", 0))

mapa_archivos_modelos = {
    "YOLOv8n":  carpeta_modelos / "yolov8n_best.pt",
    "YOLOv8m":  carpeta_modelos / "yolov8m_best.pt",
    "YOLOv11n": carpeta_modelos / "yolov11n_best.pt",
}
ruta_mejor_modelo = mapa_archivos_modelos[nombre_mejor_modelo]

print(f"Mejor modelo: {nombre_mejor_modelo} (mAP50 = {resultados_evaluacion[nombre_mejor_modelo]['map50']:.4f})")

modelo_mejor = YOLO(str(ruta_mejor_modelo))

metricas_por_clase = modelo_mejor.val(
    data=ruta_yaml_dataset,
    device=dispositivo,
    workers=num_workers,
    verbose=True,
)

In [ ]:
try:
    ap_por_clase = metricas_por_clase.box.ap50

    df_por_clase = pd.DataFrame({
        "Clase": nombres_clases[:len(ap_por_clase)],
        "AP@50": np.round(ap_por_clase, 4),
    }).sort_values("AP@50", ascending=True)

    def color_segun_clase(nombre_clase):
        if nombre_clase.startswith("NO-"):
            return "#FF5722"
        elif nombre_clase in {"Hardhat", "Mask", "Safety Vest"}:
            return "#4CAF50"
        return "#90A4AE"

    colores_barras = [color_segun_clase(c) for c in df_por_clase["Clase"]]

    fig, eje = plt.subplots(figsize=(10, 6))
    barras = eje.barh(df_por_clase["Clase"], df_por_clase["AP@50"],
                      color=colores_barras, edgecolor="white", linewidth=1)

    for barra, val in zip(barras, df_por_clase["AP@50"]):
        eje.text(val + 0.005, barra.get_y() + barra.get_height() / 2,
                 f"{val:.3f}", va="center", fontsize=9)

    parches_leyenda = [
        mpatches.Patch(color="#4CAF50", label="EPP Presente"),
        mpatches.Patch(color="#FF5722", label="EPP Ausente (incumplimiento)"),
        mpatches.Patch(color="#90A4AE", label="Otros"),
    ]
    eje.legend(handles=parches_leyenda, loc="lower right", fontsize=10)
    eje.set_xlim(0, 1.05)
    eje.set_xlabel("Average Precision @ 50 IoU")
    eje.set_title(f"AP@50 por clase - {nombre_mejor_modelo}", fontsize=13, fontweight="bold")
    eje.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig(carpeta_datos / "ap_per_class.png", bbox_inches="tight", dpi=150)
    plt.show()
except Exception as e:
    print(f"No se pudo extraer AP por clase: {e}")

### Predicciones del mejor modelo sobre imágenes de validación

In [ ]:
ruta_dataset_yaml = Path(ruta_yaml_dataset).parent
imagenes_validacion = list((ruta_dataset_yaml / "valid" / "images").glob("*.jpg"))
random.seed(2024)
muestras_validacion = random.sample(imagenes_validacion, min(6, len(imagenes_validacion)))

fig, ejes = plt.subplots(2, 3, figsize=(18, 10))
ejes = ejes.flatten()

for eje, ruta_img in zip(ejes, muestras_validacion):
    resultados_pred = modelo_mejor.predict(str(ruta_img), conf=0.4, verbose=False, device=dispositivo)
    imagen_anotada = resultados_pred[0].plot()
    imagen_rgb = cv2.cvtColor(imagen_anotada, cv2.COLOR_BGR2RGB)
    eje.imshow(imagen_rgb)
    eje.axis("off")
    cantidad_detecciones = len(resultados_pred[0].boxes)
    eje.set_title(f"{ruta_img.name[:25]}\n{cantidad_detecciones} detecciones", fontsize=8)

plt.suptitle(f"Predicciones del mejor modelo ({nombre_mejor_modelo}) - conf >= 0.4", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(carpeta_datos / "best_model_predictions.png", bbox_inches="tight", dpi=150)
plt.show()

### Justificación del modelo seleccionado

In [ ]:
print("=" * 65)
print("  ANALISIS COMPARATIVO - Sistema de Monitoreo EPP")
print("=" * 65)

for nombre_modelo in resultados_evaluacion:
    e = resultados_evaluacion[nombre_modelo]
    l = resultados_latencia.get(nombre_modelo, {})
    params = info_modelos.get(nombre_modelo, {}).get("params_M", "?")

    print(f"\n  {nombre_modelo}")
    print(f"   Parametros:   {params}M")
    print(f"   mAP50:        {e['map50']:.4f}")
    print(f"   mAP50-95:     {e['map50_95']:.4f}")
    print(f"   Precision:    {e['precision']:.4f}")
    print(f"   Recall:       {e['recall']:.4f}")
    print(f"   Latencia:     {l.get('mean_ms', '?')} ms ({l.get('fps', '?')} FPS)")

print()
print("=" * 65)
print("  CONCLUSION")
print("=" * 65)
print(f"""
Para un sistema de monitoreo de EPP en tiempo real:

- YOLOv8n: El mas rapido (menor latencia). Ideal para edge devices
  o cuando la velocidad es critica. Trade-off: menor mAP.

- YOLOv8m: El mas preciso entre los v8, pero el mas lento.
  Recomendado si se dispone de GPU y se prioriza precision.

- YOLOv11n: Nueva arquitectura con menos parametros que v8n
  pero arquitectura mejorada. Mejor balance velocidad/precision
  en arquitecturas recientes.

Modelo seleccionado para la demo: {nombre_mejor_modelo}
  (mayor mAP50 entre los entrenados)
""")

---
## Parte 4 — Tracking con ByteTrack y demo de monitoreo EPP

Se integra ByteTrack para seguimiento persistente de trabajadores en video.
Cada persona recibe un ID unico y se evalua su cumplimiento de EPP frame a frame
mediante una ventana deslizante. Cuando un trabajador permanece sin EPP durante
una fraccion significativa de la ventana, se genera una alerta.

### Arquitectura del sistema

```
Video de entrada
      |
      v
  YOLO (deteccion de EPP y personas)
      |
      v
  ByteTrack (asigna Track ID persistente por persona)
      |
      v
  Motor de compliance (evalua EPP por Track ID con ventana deslizante)
      |
      v
  Generador de alertas (alerta visual si incumplimiento persistente)
      |
      v
  Video anotado + Reporte de cumplimiento
```

### Motor de compliance EPP

Define las clases que representan EPP presente y ausente, y la lógica de evaluación
por ventana deslizante.

In [ ]:
from collections import defaultdict, deque

# Clases de EPP y violaciones
CLASES_EPP = {
    "presente": {"Hardhat", "Mask", "Safety Vest"},
    "ausente":  {"NO-Hardhat", "NO-Mask", "NO-Safety Vest"},
}

# Colores de visualizacion
COLORES_ESTADO = {
    "cumple":       (50,  205, 50),    # verde
    "no_cumple":    (220, 50,  50),    # rojo
    "indeterminado": (180, 180, 180),  # gris
}


class MonitorCumplimientoEPP:
    """
    Mantiene el estado de cumplimiento EPP por Track ID.
    Usa una ventana deslizante de N frames para determinar
    si un trabajador esta incumpliendo de forma persistente.
    """
    def __init__(self, tamanio_ventana: int = 30, umbral_alerta: float = 0.7):
        self.tamanio_ventana = tamanio_ventana
        self.umbral_alerta = umbral_alerta
        self.historial = defaultdict(lambda: deque(maxlen=tamanio_ventana))
        self.violaciones = defaultdict(set)
        self.total_alertas = 0
        self.ids_alertados = set()

    def actualizar(self, track_id: int, clases_detectadas: list) -> dict:
        """
        Actualiza el estado de un trabajador dado su track_id
        y las clases detectadas en el frame actual.
        """
        epp_faltante = CLASES_EPP["ausente"].intersection(set(clases_detectadas))
        cumple = len(epp_faltante) == 0

        self.historial[track_id].append(cumple)
        self.violaciones[track_id] = epp_faltante

        if len(self.historial[track_id]) < self.tamanio_ventana // 3:
            estado = "indeterminado"
        else:
            ratio_incumplimiento = 1 - (sum(self.historial[track_id]) / len(self.historial[track_id]))
            if ratio_incumplimiento >= self.umbral_alerta:
                estado = "no_cumple"
                if track_id not in self.ids_alertados:
                    self.total_alertas += 1
                    self.ids_alertados.add(track_id)
            else:
                estado = "cumple"
                self.ids_alertados.discard(track_id)

        return {"estado": estado, "epp_faltante": epp_faltante}

    def obtener_tasa_cumplimiento(self) -> float:
        if not self.historial:
            return 1.0
        todos_los_frames = [v for hist in self.historial.values() for v in hist]
        return sum(todos_los_frames) / len(todos_los_frames) if todos_los_frames else 1.0


print("MonitorCumplimientoEPP definido")

### Pipeline de procesamiento de video con ByteTrack

In [ ]:
def dibujar_overlay_cumplimiento(frame: np.ndarray, track_id: int, bbox,
                                  estado: str, epp_faltante: set, confianza: float) -> np.ndarray:
    """Dibuja el bounding box de la persona con color segun estado de cumplimiento."""
    x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    color = COLORES_ESTADO[estado]

    grosor = 3 if estado == "no_cumple" else 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, grosor)

    etiqueta = f"ID:{track_id}  {confianza:.0%}"
    (tw, th), _ = cv2.getTextSize(etiqueta, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, etiqueta, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    if epp_faltante:
        desplazamiento_y = y1 + 20
        for item in epp_faltante:
            texto_alerta = f"FALTA: {item}"
            cv2.putText(frame, texto_alerta, (x1 + 5, desplazamiento_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
            desplazamiento_y += 18

    return frame


def agregar_panel_estadisticas(frame: np.ndarray, num_frame: int, total_tracks: int,
                                cumplen: int, no_cumplen: int, tasa_cumplimiento: float) -> np.ndarray:
    """Panel de estadisticas en la esquina superior izquierda."""
    h, w = frame.shape[:2]
    overlay = frame.copy()

    cv2.rectangle(overlay, (10, 10), (280, 120), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    lineas_stats = [
        f"Frame: {num_frame:04d}",
        f"Trabajadores: {total_tracks}",
        f"Cumplen EPP: {cumplen}",
        f"Incumplen:   {no_cumplen}",
        f"Compliance:  {tasa_cumplimiento:.1%}",
    ]

    colores_lineas = [(255, 255, 255), (255, 255, 255), (0, 200, 0), (0, 0, 255), (255, 255, 255)]
    for i, (texto, color_texto) in enumerate(zip(lineas_stats, colores_lineas)):
        cv2.putText(frame, texto, (18, 32 + i * 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_texto, 1)

    if no_cumplen > 0:
        alerta = f"ALERTA: {no_cumplen} trabajador(es) sin EPP!"
        (aw, ah), _ = cv2.getTextSize(alerta, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
        cv2.rectangle(frame, (w // 2 - aw // 2 - 10, 5), (w // 2 + aw // 2 + 10, 35), (0, 0, 180), -1)
        cv2.putText(frame, alerta, (w // 2 - aw // 2, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    return frame


print("Funciones de visualizacion definidas")

In [ ]:
def procesar_video_con_tracking(
    video_entrada: str,
    video_salida: str,
    ruta_modelo: str,
    umbral_confianza: float = 0.4,
    max_frames: int = None,
) -> dict:
    """
    Procesa un video aplicando deteccion YOLO + ByteTrack.
    Retorna un diccionario con estadisticas del video procesado.
    """
    modelo = YOLO(ruta_modelo)
    monitor = MonitorCumplimientoEPP(tamanio_ventana=30, umbral_alerta=0.6)

    cap = cv2.VideoCapture(video_entrada)
    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {video_entrada}")

    fps_video = cap.get(cv2.CAP_PROP_FPS) or 25
    ancho_video = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto_video = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video: {ancho_video}x{alto_video} @ {fps_video:.1f} FPS  |  {total_frames} frames")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    escritor = cv2.VideoWriter(video_salida, fourcc, fps_video, (ancho_video, alto_video))

    num_frame = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and num_frame >= max_frames:
            break

        resultados = modelo.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=umbral_confianza,
            device=dispositivo,
            verbose=False,
        )

        frame_anotado = frame.copy()
        estado_actual = {}

        if resultados[0].boxes is not None and resultados[0].boxes.id is not None:
            cajas = resultados[0].boxes.xyxy.cpu().numpy()
            ids_track = resultados[0].boxes.id.cpu().numpy().astype(int)
            ids_clase = resultados[0].boxes.cls.cpu().numpy().astype(int)
            confianzas = resultados[0].boxes.conf.cpu().numpy()

            detecciones_por_track = defaultdict(list)
            cajas_por_track = {}
            confianza_por_track = {}

            for caja, tid, cid, conf in zip(cajas, ids_track, ids_clase, confianzas):
                nombre_clase = nombres_clases[cid]
                detecciones_por_track[tid].append(nombre_clase)
                if tid not in cajas_por_track or conf > confianza_por_track[tid]:
                    cajas_por_track[tid] = caja
                    confianza_por_track[tid] = conf

            conteo_cumplen = conteo_no_cumplen = 0

            for tid, clases_det in detecciones_por_track.items():
                if "Person" not in clases_det:
                    continue

                resultado = monitor.actualizar(tid, clases_det)
                estado = resultado["estado"]
                faltante = resultado["epp_faltante"]
                estado_actual[tid] = estado

                if estado == "cumple":
                    conteo_cumplen += 1
                elif estado == "no_cumple":
                    conteo_no_cumplen += 1

                if tid in cajas_por_track:
                    dibujar_overlay_cumplimiento(
                        frame_anotado, tid, cajas_por_track[tid],
                        estado, faltante, confianza_por_track[tid]
                    )

            frame_anotado = agregar_panel_estadisticas(
                frame_anotado, num_frame,
                len(detecciones_por_track),
                conteo_cumplen, conteo_no_cumplen,
                monitor.obtener_tasa_cumplimiento()
            )

        escritor.write(frame_anotado)
        num_frame += 1

        if num_frame % 50 == 0:
            print(f"  Procesado: {num_frame}/{total_frames if total_frames else '?'} frames"
                  f"  | Tracks activos: {len(estado_actual)}")

    cap.release()
    escritor.release()

    estadisticas = {
        "frames_procesados": num_frame,
        "tracks_unicos": len(monitor.historial),
        "total_alertas": monitor.total_alertas,
        "tasa_cumplimiento": round(monitor.obtener_tasa_cumplimiento(), 4),
        "video_salida": video_salida,
    }
    print(f"\nVideo guardado: {video_salida}")
    print(f"   Frames:         {estadisticas['frames_procesados']}")
    print(f"   Tracks unicos:  {estadisticas['tracks_unicos']}")
    print(f"   Alertas:        {estadisticas['total_alertas']}")
    print(f"   Compliance:     {estadisticas['tasa_cumplimiento']:.1%}")
    return estadisticas


print("Pipeline de tracking definido")

### Descargar video de muestra y ejecutar demo

Se descarga un video de obra de construcción y se procesan los primeros 300 frames
como demostración del sistema completo.

In [ ]:
import subprocess
import os

carpeta_videos = Path("videos")
carpeta_videos.mkdir(exist_ok=True)

video_entrada = str(carpeta_videos / "input.mp4")

if not os.path.exists(video_entrada):
    print("Instalando yt-dlp...")
    subprocess.run(["pip", "install", "-q", "yt-dlp"], capture_output=True)
    print("Descargando video de muestra...")
    url_video = "https://www.youtube.com/watch?v=p-rSdt0aFuw"
    resultado_descarga = subprocess.run([
        "yt-dlp",
        "-f", "best[height<=480]",
        "-o", video_entrada,
        "--max-filesize", "50m",
        url_video
    ], capture_output=True, text=True)
    if resultado_descarga.returncode == 0:
        print(f"Video descargado: {video_entrada}")
    else:
        print(f"Error descargando video: {resultado_descarga.stderr}")
        print("Alternativa: colocar manualmente un video en videos/input.mp4")
else:
    print(f"Video de entrada encontrado: {video_entrada}")

In [ ]:
video_salida = str(carpeta_videos / "output_tracked.mp4")

if not Path(video_entrada).exists():
    print("No hay video de entrada. Colocar un video en videos/input.mp4")
else:
    estadisticas_demo = procesar_video_con_tracking(
        video_entrada=video_entrada,
        video_salida=video_salida,
        ruta_modelo=str(ruta_mejor_modelo),
        umbral_confianza=0.4,
        max_frames=300,
    )

    with open(carpeta_datos / "demo_stats.json", "w") as f:
        json.dump(estadisticas_demo, f, indent=2)
    print("\nEstadisticas guardadas en data/demo_stats.json")

### Preview del video procesado

In [ ]:
def extraer_frames_video(ruta_video: str, n_frames: int = 6,
                          ratio_inicio: float = 0.1) -> list:
    """Extrae N frames distribuidos del video para visualizacion."""
    cap = cv2.VideoCapture(ruta_video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(int(total * ratio_inicio), total - 1, n_frames, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


if Path(video_salida).exists():
    frames_preview = extraer_frames_video(video_salida, n_frames=6)

    fig, ejes = plt.subplots(2, 3, figsize=(18, 10))
    ejes = ejes.flatten()

    for i, (eje, frame) in enumerate(zip(ejes, frames_preview)):
        eje.imshow(frame)
        eje.axis("off")
        eje.set_title(f"Frame {i + 1}", fontsize=10)

    plt.suptitle(f"Preview del video procesado - {nombre_mejor_modelo} + ByteTrack",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(carpeta_datos / "demo_preview.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("Video de salida no encontrado. Ejecutar el demo primero.")

### Reporte final del sistema

In [ ]:
print("=" * 65)
print("  REPORTE FINAL - Sistema de Monitoreo EPP")
print("=" * 65)

e = resultados_evaluacion[nombre_mejor_modelo]
print(f"""
MODELO SELECCIONADO: {nombre_mejor_modelo}
-------------------------------------
Metricas de deteccion (Construction-PPE Dataset):
  mAP@50:        {e['map50']:.4f}
  mAP@50-95:     {e['map50_95']:.4f}
  Precision:     {e['precision']:.4f}
  Recall:        {e['recall']:.4f}

Componentes del sistema:
  - Detector:    {nombre_mejor_modelo} fine-tuneado en Construction-PPE
  - Tracker:     ByteTrack (Ultralytics)
  - Compliance:  Ventana deslizante de 30 frames / umbral 60%
  - Alertas:     Visual en tiempo real + log por Track ID

Clases monitoreadas:
  EPP Presente:  Hardhat | Mask | Safety Vest
  EPP Ausente:   NO-Hardhat | NO-Mask | NO-Safety Vest
""")

ruta_stats_demo = carpeta_datos / "demo_stats.json"
if ruta_stats_demo.exists():
    with open(ruta_stats_demo) as f:
        ds = json.load(f)
    print(f"""Resultado del demo:
  Frames procesados: {ds['frames_procesados']}
  Trabajadores detectados (tracks unicos): {ds['tracks_unicos']}
  Alertas de incumplimiento: {ds['total_alertas']}
  Tasa de cumplimiento global: {ds['tasa_cumplimiento']:.1%}
""")

print("=" * 65)

### Descarga de todos los artefactos generados

In [ ]:
from google.colab import files

archivos_a_descargar = [
    # Modelos entrenados
    carpeta_modelos / "yolov8n_best.pt",
    carpeta_modelos / "yolov8m_best.pt",
    carpeta_modelos / "yolov11n_best.pt",
    # Metricas y resultados
    carpeta_datos / "eval_results.json",
    carpeta_datos / "latency_results.json",
    carpeta_datos / "dataset_config.json",
    carpeta_datos / "demo_stats.json",
    # Graficos
    carpeta_datos / "class_distribution.png",
    carpeta_datos / "bbox_stats.png",
    carpeta_datos / "sample_annotations.png",
    carpeta_datos / "comparison_metrics.png",
    carpeta_datos / "speed_accuracy_tradeoff.png",
    carpeta_datos / "training_curves.png",
    carpeta_datos / "ap_per_class.png",
    carpeta_datos / "best_model_predictions.png",
    carpeta_datos / "demo_preview.png",
    # Video procesado
    Path(video_salida),
]

for archivo in archivos_a_descargar:
    if archivo.exists():
        print(f"Descargando {archivo.name}...")
        files.download(str(archivo))
    else:
        print(f"  {archivo.name} no encontrado, se omite.")